# Biosignal capture workbench

Select **Python 3 (ipykernel)** and run the launch cell with **Shift+Enter**. Simulation is selected by default. Stop an ongoing capture before reloading the interface.

[Project overview](README.md) · [Hardware and firmware](docs/HARDWARE.md) · [Algorithms and parameter versions](docs/ALGORITHMS.md)


## Step 1: Check the environment

Run the next cell. Environment ready confirms that the tools for this experiment have loaded.


In [ ]:
import sys
from pathlib import Path
import csv
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pressure_demo import collect_pressure, save_csv, preview_html

ROOT = Path.cwd()
assert (ROOT / "pressure_demo.py").exists(), 'Run this notebook from the sensor_demos directory.'
print('Environment ready. Python: ', sys.version.split()[0])
print('Data source: simulation; no real hardware is being read.')

## Step 2: Set up this experiment

- `DURATION_S = 10`: capture for about 10 seconds.
- `SAMPLE_HZ = 20`: aim for 20 readings per second. Computer scheduling may add small delays.
- Automatic demo: released for the first 3 seconds, pressed from seconds 3 to 7, then released.
- Manual slider: select it before running Step 3. Drag the slider during capture to observe the numbers and curve.

0–4095 is the simulated ADC range for this experiment, not a pressure unit. The slider is not connected to a real sensor.


In [ ]:
DURATION_S = 10
SAMPLE_HZ = 20
mode = widgets.Dropdown(
    options=[('Automatic press and release demo', "automatic"), ('Simulate pressure with the slider', "manual")],
    description='Demo mode: ', style={"description_width": "initial"})
pressure_slider = widgets.IntSlider(
    value=300, min=0, max=4095, step=1, description='Virtual pressure: ',
    continuous_update=True, style={"description_width": "initial"},
    layout=widgets.Layout(width="650px", max_width="100%"))
stop_state = {"requested": False}
stop_button = widgets.Button(description='Stop this capture', disabled=True)
def request_stop(_):
    stop_state["requested"] = True
stop_button.on_click(request_stop)
status = widgets.HTML('<b>Ready. Run Step 3.</b>')
display(widgets.VBox([mode, pressure_slider, stop_button, status]))

## Step 3: Read continuously and display live

The loop runs for about 10 seconds. During capture, drag the slider if manual mode is selected, or click Stop this capture.

`collect_pressure` produces a simulated sample about every 1/20 second. `update_display` refreshes the chart about twice per second.
Sampling and plotting can run at different rates; every sample is still saved.

To repeat, change the mode in Step 2 and run this cell again. The code creates a new recording.


In [ ]:
rows = []
stop_state["requested"] = False
stop_button.disabled = False
mode.disabled = True
scenario = mode.value
status.value = '<b>Capturing simulated data...</b>'

with plt.ioff():
    fig, ax = plt.subplots(figsize=(9, 3.5))
line, = ax.plot([], [], color="#127b72", linewidth=2)
ax.set(xlim=(0, DURATION_S), ylim=(0, 4095),
       xlabel="Time (s)", ylabel="Simulated ADC count",
       title="SIMULATED pressure: release / press / release")
ax.grid(alpha=0.2)
fig.tight_layout()
chart = display(fig, display_id=True)

def update_display(current_rows):
    if not current_rows:
        return
    last = current_rows[-1]
    line.set_data([r["elapsed_s"] for r in current_rows],
                  [r["pressure_adc"] for r in current_rows])
    chart.update(fig)
    status.value = (f"<b>Simulated data</b> | Samples read: {len(current_rows)} | Time: {last['elapsed_s']:.1f} s | Current value: {last['pressure_adc']}")

try:
    await collect_pressure(
        duration_s=DURATION_S,
        sample_hz=SAMPLE_HZ,
        manual_reader=(lambda: pressure_slider.value) if scenario == "manual" else None,
        on_update=update_display,
        should_stop=lambda: stop_state["requested"],
        records=rows,
    )
finally:
    plt.close(fig)
    stop_button.disabled = True
    mode.disabled = False
    status.value += '<br><b>Capture finished. Run Step 4 to save.</b>'

print(f'This capture contains {len(rows)} simulated samples. Continue to Step 4.')

## Step 4: Save and reopen the file to check it

A CSV is a simple table with one reading per row. The first five rows are displayed after saving.
The filename and every row contain `SIMULATED` to distinguish them from future real measurements.

Success confirms this small workflow: generate numbers → display continuously → save to the computer.


In [ ]:
if rows:
    csv_path = save_csv(rows, ROOT / "data")
    with csv_path.open(encoding="utf-8-sig", newline="") as file:
        saved_rows = list(csv.DictReader(file))
    assert len(saved_rows) == len(rows), 'The saved sample count does not match the captured sample count.'
    assert all(row["source"] == "SIMULATED" for row in saved_rows)
    print('Saved successfully: ', csv_path.name)
    print('Reopened and verified: ', len(saved_rows), 'rows; all are labelled as simulated.')
    display(HTML(preview_html(saved_rows)))
    png_path = csv_path.with_suffix(".png")
    fig.savefig(png_path, dpi=150)
    print('The same curve was also saved as an image: ', png_path.name)
else:
    print('No recording yet. Run Step 3 again and capture at least one sample before saving.')

## The first demo is complete

You can now explain how many readings are taken per second, what each reading contains and where the data is saved.

This experiment verifies the simulated workflow only. It does not confirm that a real sensor works or infer breathing, tension or panic from pressure data.

When the hardware arrives, the next demo separately verifies ESP32 firmware upload and serial transmission before connecting the film pressure sensor.
ESP32 adaptation of the old HeartSpeed library is left for the ECG demo, so each part can be understood separately.
